# FinBERT Multitask Prototype — Historical Experiment

This notebook documents the first working multitask FinBERT prototype used during Aroogula's model-development phase.

It fine-tunes `ProsusAI/finbert` for two related tasks:

1. **Sentiment classification** — positive / negative / neutral.
2. **Tradeability classification** — whether a news item is a plausible trading catalyst.

> **Status:** historical experiment. The production-oriented training workflow is documented in
> `02_finbert_multitask_training.ipynb`, which supersedes this notebook.

The notebook is retained to show the evolution from the initial prototype to the current training pipeline.

## 1. Configuration and dataset

The cleaned repository expects the training workbook at:

`../data/training/FINBERT_V3_TRAINING_DATA.xlsx`

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import accuracy_score
from sklearn.utils.class_weight import compute_class_weight

from torch import nn
from torch.utils.data import Dataset

from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)

DATA_PATH = Path("../data/training/FINBERT_V3_TRAINING_DATA.xlsx")
SHEET_NAME = "TRAIN_V3"
MODEL_NAME = "ProsusAI/finbert"
MODEL_OUTPUT_DIR = Path("../artifacts/finbert_multitask_prototype")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Training dataset not found at {DATA_PATH.resolve()}. "
        "Place FINBERT_V3_TRAINING_DATA.xlsx under data/training/."
    )

training_data = pd.read_excel(DATA_PATH, sheet_name=SHEET_NAME)

train_df = training_data[
    training_data["split"].astype(str).str.lower() == "train"
].reset_index(drop=True)

validation_df = training_data[
    training_data["split"].astype(str).str.lower() == "validation"
].reset_index(drop=True)

test_df = training_data[
    training_data["split"].astype(str).str.lower() == "test"
].reset_index(drop=True)

print(f"Train: {len(train_df):,}")
print(f"Validation: {len(validation_df):,}")
print(f"Test: {len(test_df):,}")

## 2. PyTorch dataset

Each example carries separate labels and optional sample weights for sentiment and tradeability.

In [ ]:
class FinBertDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, tokenizer):
        self.texts = dataframe["text"].fillna("").astype(str).tolist()
        self.sentiment = dataframe["finbert_sentiment"].to_numpy()
        self.tradeable = dataframe["tradeable"].to_numpy()
        self.sentiment_weights = dataframe["sentiment_weight"].to_numpy()
        self.tradeability_weights = dataframe["tradeability_weight"].to_numpy()
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            max_length=256,
            return_tensors="pt",
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.sentiment[idx], dtype=torch.long),
            "tradeable": torch.tensor(self.tradeable[idx], dtype=torch.long),
            "sentiment_weight": torch.tensor(
                self.sentiment_weights[idx],
                dtype=torch.float,
            ),
            "tradeability_weight": torch.tensor(
                self.tradeability_weights[idx],
                dtype=torch.float,
            ),
        }


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_dataset = FinBertDataset(train_df, tokenizer)
validation_dataset = FinBertDataset(validation_df, tokenizer)
test_dataset = FinBertDataset(test_df, tokenizer)

## 3. Multitask model

The original FinBERT encoder and sentiment head initialize the sentiment task. A second classification head is added for tradeability.

In [ ]:
class MultiTaskFinBert(nn.Module):
    def __init__(self):
        super().__init__()

        original_finbert = AutoModelForSequenceClassification.from_pretrained(
            MODEL_NAME
        )

        self.bert = original_finbert.bert
        self.dropout = original_finbert.dropout

        hidden_size = original_finbert.config.hidden_size

        self.sentiment = nn.Linear(hidden_size, 3)
        self.sentiment.load_state_dict(
            original_finbert.classifier.state_dict()
        )

        self.tradeable = nn.Linear(hidden_size, 2)

    def forward(self, input_ids, attention_mask, **kwargs):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        pooled_output = self.dropout(outputs.pooler_output)

        return (
            self.sentiment(pooled_output),
            self.tradeable(pooled_output),
        )


model = MultiTaskFinBert()

## 4. Weighted multitask loss

The prototype optimizes both tasks jointly:

`total_loss = sentiment_loss + lambda_tradeability * tradeability_loss`

Per-example weights are normalized so that a batch with larger average weights does not automatically change the overall loss scale.

In [ ]:
class WeightedTrainer(Trainer):
    def __init__(
        self,
        lambda_tradeability: float = 1.0,
        sentiment_class_weights=None,
        tradeability_class_weights=None,
        *args,
        **kwargs,
    ):
        super().__init__(*args, **kwargs)
        self.lambda_tradeability = float(lambda_tradeability)
        self.sentiment_class_weights = sentiment_class_weights
        self.tradeability_class_weights = tradeability_class_weights

    @staticmethod
    def weighted_mean(losses, weights):
        if weights is None:
            return losses.mean()

        weights = weights.to(
            device=losses.device,
            dtype=losses.dtype,
        )

        return (
            (losses * weights).sum()
            / weights.sum().clamp_min(1e-8)
        )

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        **kwargs,
    ):
        labels_sentiment = inputs.pop("labels")
        labels_tradeability = inputs.pop("tradeable")

        sentiment_weights = inputs.pop("sentiment_weight", None)
        tradeability_weights = inputs.pop("tradeability_weight", None)

        sentiment_logits, tradeability_logits = model(**inputs)

        sentiment_class_weights = self.sentiment_class_weights
        tradeability_class_weights = self.tradeability_class_weights

        if sentiment_class_weights is not None:
            sentiment_class_weights = sentiment_class_weights.to(
                sentiment_logits.device
            )

        if tradeability_class_weights is not None:
            tradeability_class_weights = tradeability_class_weights.to(
                tradeability_logits.device
            )

        sentiment_loss_fn = nn.CrossEntropyLoss(
            weight=sentiment_class_weights,
            reduction="none",
        )
        tradeability_loss_fn = nn.CrossEntropyLoss(
            weight=tradeability_class_weights,
            reduction="none",
        )

        sentiment_loss = self.weighted_mean(
            sentiment_loss_fn(
                sentiment_logits,
                labels_sentiment,
            ),
            sentiment_weights,
        )

        tradeability_loss = self.weighted_mean(
            tradeability_loss_fn(
                tradeability_logits,
                labels_tradeability,
            ),
            tradeability_weights,
        )

        total_loss = (
            sentiment_loss
            + self.lambda_tradeability * tradeability_loss
        )

        outputs = {
            "sentiment_logits": sentiment_logits,
            "tradeability_logits": tradeability_logits,
        }

        return (total_loss, outputs) if return_outputs else total_loss

## 5. Training configuration

This historical version used class balancing plus the per-example weights embedded in the training workbook. The later V3 notebook evaluates the two tasks more carefully and calibrates the tradeability decision threshold.

In [ ]:
sentiment_class_weights = torch.tensor(
    compute_class_weight(
        class_weight="balanced",
        classes=np.array([0, 1, 2]),
        y=train_df["finbert_sentiment"].to_numpy(),
    ),
    dtype=torch.float,
)

tradeability_class_weights = torch.tensor(
    compute_class_weight(
        class_weight="balanced",
        classes=np.array([0, 1]),
        y=train_df["tradeable"].to_numpy(),
    ),
    dtype=torch.float,
)


def compute_metrics(eval_pred):
    sentiment_logits, tradeability_logits = eval_pred.predictions
    sentiment_labels, tradeability_labels = eval_pred.label_ids

    sentiment_predictions = np.argmax(sentiment_logits, axis=-1)
    tradeability_predictions = np.argmax(tradeability_logits, axis=-1)

    sentiment_accuracy = accuracy_score(
        sentiment_labels,
        sentiment_predictions,
    )
    tradeability_accuracy = accuracy_score(
        tradeability_labels,
        tradeability_predictions,
    )

    return {
        "sentiment_accuracy": sentiment_accuracy,
        "tradeability_accuracy": tradeability_accuracy,
        "combined_score": (
            sentiment_accuracy + tradeability_accuracy
        ) / 2.0,
    }


training_args = TrainingArguments(
    output_dir=str(MODEL_OUTPUT_DIR),
    overwrite_output_dir=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    label_names=["labels", "tradeable"],
    learning_rate=2e-5,
    weight_decay=0.01,
    num_train_epochs=5,
    warmup_ratio=0.06,
    remove_unused_columns=False,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=0,
    report_to=[],
    seed=42,
)

## 6. Train and evaluate

Training artifacts are written outside the notebook directory under `../artifacts/`.

In [ ]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    lambda_tradeability=1.0,
    sentiment_class_weights=sentiment_class_weights,
    tradeability_class_weights=tradeability_class_weights,
)

trainer.train()

validation_results = trainer.evaluate(
    eval_dataset=validation_dataset,
    metric_key_prefix="validation",
)

test_results = trainer.evaluate(
    eval_dataset=test_dataset,
    metric_key_prefix="test",
)

print("Validation:", validation_results)
print("Test:", test_results)

trainer.save_model(str(MODEL_OUTPUT_DIR / "best_model"))
tokenizer.save_pretrained(str(MODEL_OUTPUT_DIR / "best_model"))

## 7. What changed after this prototype?

The follow-up notebook (`02_finbert_multitask_training.ipynb`) improves the experiment by:

- preserving the workbook's train / validation / test split;
- evaluating macro-F1 and per-class sentiment performance;
- measuring precision, recall and F1 for tradeability;
- calibrating the tradeability threshold on validation data;
- saving deployment metadata alongside the model;
- using a cleaner `PreTrainedModel` implementation for serialization and reload.

The obsolete single-task TradeBERT experiment and the old `Engine/` integration cells were intentionally removed from this public notebook because they are no longer part of Aroogula's current architecture.